# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iemanmalik/flyrank-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [4]:
from google.colab import userdata
from datasets import load_dataset

HF_TOKEN = userdata.get("HF_TOKEN")

dataset = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",   # <-- REQUIRED
    split="train",
    token=HF_TOKEN
)

dataset

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Generating train split:   0%|          | 0/78835655 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/39 [00:00<?, ?it/s]

Dataset({
    features: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events'],
    num_rows: 78835655
})

In [5]:
march = dataset.filter(
    lambda x: (
        x["report_date"].year == 2026 and
        x["report_date"].month == 3
    )
)

march

Filter:   0%|          | 0/78835655 [00:00<?, ? examples/s]

Dataset({
    features: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events'],
    num_rows: 9841378
})

In [6]:
sample = march.shuffle(seed=42).select(range(10000))

df = sample.to_pandas()

print(df.shape)

df.head()

(10000, 30)


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2026-03-06,client_62f4a7e64f5e0096,content_727244e393338147,True,False,False,None,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-03-04,client_3197e6291363b4db,content_454a1114bc00a0a5,True,True,False,False,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2026-03-31,client_f623b01661d4bfe4,content_a71e5aa975e2b8ad,True,True,False,False,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2026-03-16,client_2b4306c3ed003f01,content_894eba4ea504ab7c,True,False,False,None,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-03-10,client_65de48885f4ef01b,content_76cfc7bd73f5f9b7,True,True,False,False,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [7]:
import pandas as pd

volume_table = (
    df.assign(volume_bucket=pd.cut(
        df["gsc_impressions"],
        bins=[-1,0,10,100,1000,100000],
        labels=[
            "0",
            "1-10",
            "11-100",
            "101-1000",
            "1000+"
        ]
    ))
    .groupby("volume_bucket")
    .size()
    .reset_index(name="n")
)

print(volume_table)

  volume_bucket     n
0             0  6296
1          1-10  1585
2        11-100  1458
3      101-1000   630
4         1000+    31


/tmp/ipykernel_15201/3556723697.py:15: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("volume_bucket")


Signal 1: Search Volume
Verdict: CONFIRMED
Pages with higher impressions are more common in the dataset and volume is an important signal when deciding which pages deserve review.

In [4]:
from google.colab import userdata
from datasets import load_dataset

HF_TOKEN = userdata.get("HF_TOKEN")

dataset = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train",
    token=HF_TOKEN
)

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 90.3MB            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 7.12MB            

...

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  146MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/78835655 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/39 [00:00<?, ?it/s]

In [5]:
march = dataset.filter(
    lambda x: (
        x["report_date"].year == 2026 and
        x["report_date"].month == 3
    )
)

Filter:   0%|          | 0/78835655 [00:00<?, ? examples/s]

In [7]:
sample = march.shuffle(seed=42).select(range(10000))

df = sample.to_pandas()

print(df.shape)

(10000, 30)


In [9]:
from google.colab import userdata
from datasets import load_dataset
import pandas as pd

HF_TOKEN = userdata.get("HF_TOKEN")

dataset = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train",
    token=HF_TOKEN
)

march = dataset.filter(
    lambda x: (
        x["report_date"].year == 2026 and
        x["report_date"].month == 3
    )
)

sample = march.shuffle(seed=42).select(range(10000))

df = sample.to_pandas()

print(df.shape)
df.head()

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/39 [00:00<?, ?it/s]

(10000, 30)


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2026-03-06,client_62f4a7e64f5e0096,content_727244e393338147,True,False,False,None,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-03-04,client_3197e6291363b4db,content_454a1114bc00a0a5,True,True,False,False,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2026-03-31,client_f623b01661d4bfe4,content_a71e5aa975e2b8ad,True,True,False,False,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2026-03-16,client_2b4306c3ed003f01,content_894eba4ea504ab7c,True,False,False,None,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-03-10,client_65de48885f4ef01b,content_76cfc7bd73f5f9b7,True,True,False,False,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [10]:
df.columns

Index(['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc',
       'client_has_ga4', 'gsc_data_available', 'ga4_data_available',
       'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position',
       'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions',
       'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct',
       'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai',
       'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude',
       'ai_meta', 'ai_other', 'scroll_events'],
      dtype='object')

## Signal Check 2: Click-Through Rate (CTR)

Signal used: CTR

Bucket Table:

- <1% CTR: 3496 pages
- 1–5% CTR: 175 pages
- 5–10% CTR: 21 pages
- >10% CTR: 12 pages

n = 3704 pages with valid CTR values.

Verdict: CONFIRMED

Reason:
Most pages have very low CTR, indicating that click-through rate varies considerably across content. CTR is therefore a useful signal for identifying pages that may need title, meta description, or content improvements.

In [11]:
df["ctr"] = (
    df["gsc_clicks"] /
    df["gsc_impressions"].replace(0, pd.NA)
)

In [13]:
import numpy as np

# Create CTR safely
df["ctr"] = np.where(
    df["gsc_impressions"] > 0,
    df["gsc_clicks"] / df["gsc_impressions"],
    np.nan
)

In [14]:
ctr_table = (
    df.dropna(subset=["ctr"])
      .assign(
          ctr_bucket=pd.cut(
              df.dropna(subset=["ctr"])["ctr"],
              bins=[0, 0.01, 0.05, 0.10, 1],
              labels=["<1%", "1-5%", "5-10%", ">10%"],
              include_lowest=True
          )
      )
      .groupby("ctr_bucket", observed=True)
      .size()
      .reset_index(name="n")
)

print(ctr_table)

  ctr_bucket     n
0        <1%  3496
1       1-5%   175
2      5-10%    21
3       >10%    12


In [15]:
import numpy as np

# Visibility Score (0-100)
df["visibility_score"] = (
    df["gsc_impressions"] / df["gsc_impressions"].max()
) * 100

# Position Opportunity
df["position_score"] = np.where(
    df["gsc_avg_position"].notna(),
    100 - df["gsc_avg_position"],
    0
)

# Final Baseline Score
df["baseline_score"] = (
    0.7 * df["visibility_score"] +
    0.3 * df["position_score"]
)

In [16]:
df["reason_code"] = np.where(
    df["gsc_impressions"] >= 500,
    "high_visibility_page",
    "low_visibility_page"
)

In [17]:
df["action"] = np.where(
    df["baseline_score"] >= 50,
    "Review",
    "Monitor"
)

In [18]:
queue = df.sort_values(
    "baseline_score",
    ascending=False
)

queue[
    [
        "content_hash_id",
        "gsc_impressions",
        "gsc_avg_position",
        "baseline_score",
        "reason_code",
        "action"
    ]
].head(10)

,content_hash_id,gsc_impressions,gsc_avg_position,baseline_score,reason_code,action
9394,content_93aadc6eeccba7b8,3005,5.794676,98.261597,high_visibility_page,Review
8148,content_306bc78dff1eb683,2707,2.933875,92.178074,high_visibility_page,Review
8160,content_e8b074fd4a082388,2277,5.572683,81.369792,high_visibility_page,Review
8672,content_9f1f611e98240777,2547,35.173930,78.778936,high_visibility_page,Review
6465,content_c19eed2225ee5f40,2007,2.920777,75.875847,high_visibility_page,Review
1445,content_4231c23af6ad15e9,1938,4.133643,73.904666,high_visibility_page,Review
9869,content_8eba2d99238592aa,1890,4.737037,72.605511,high_visibility_page,Review
4593,content_26461136a15df002,1841,2.414992,72.160694,high_visibility_page,Review
4684,content_29caed0f85034d5c,1799,5.638132,70.215382,high_visibility_page,Review
4182,content_6478f978b79a1b49,1603,2.540861,66.578840,high_visibility_page,Review


In [19]:
import os

os.makedirs("work/outputs", exist_ok=True)

queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("baseline_action_score.csv saved successfully!")

baseline_action_score.csv saved successfully!


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.